In [1]:
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score, log_loss

In [2]:
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("mnist-mlp")

2026/08/30 18:05:44 INFO mlflow.tracking.fluent: Experiment with name 'mnist-mlp' does not exist. Creating a new experiment.


<Experiment: artifact_location='/home/sriharsha/projects/DA3408_DA24B034_1/q2/mlruns/1', creation_time=1788093344928, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1788093344928, lifecycle_stage='active', name='mnist-mlp', tags={}, trace_location=None, workspace='default'>

In [3]:
X, y = fetch_openml("mnist_784", version=1, return_X_y=True, as_frame=False)
X = X / 255.0  
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
def train_and_log(hidden_layer_sizes, learning_rate_init, batch_size, run_name):
    with mlflow.start_run(run_name=run_name):
       
        mlflow.log_param("hidden_layer_sizes", hidden_layer_sizes)
        mlflow.log_param("learning_rate_init", learning_rate_init)
        mlflow.log_param("batch_size", batch_size)
        mlflow.log_param("model_type", "MLPClassifier")
        mlflow.log_param("dataset", "MNIST")

        model = MLPClassifier(
            hidden_layer_sizes=hidden_layer_sizes,
            learning_rate_init=learning_rate_init,
            batch_size=batch_size,
            max_iter=30,
            early_stopping=True,
            random_state=42,
        )
        model.fit(X_train, y_train)

        train_preds = model.predict(X_train)
        val_preds = model.predict(X_test)

        train_loss = log_loss(y_train, model.predict_proba(X_train))
        val_accuracy = accuracy_score(y_test, val_preds)
        val_f1 = f1_score(y_test, val_preds, average="macro")

        mlflow.log_metric("train_loss", train_loss)
        mlflow.log_metric("val_accuracy", val_accuracy)
        mlflow.log_metric("val_f1_macro", val_f1)
        mlflow.log_metric("n_iter", model.n_iter_)

        signature = infer_signature(X_train[:5], model.predict(X_train[:5]))
        mlflow.sklearn.log_model(model,name="model",signature=signature,
        input_example=X_train[:5],skops_trusted_types=[
        "sklearn.neural_network._stochastic_optimizers.AdamOptimizer"
    ]
)

        run_id = mlflow.active_run().info.run_id
        print(f"{run_name}: train_loss={train_loss:.4f} val_acc={val_accuracy:.4f}")
        return run_id

In [5]:
configs = [
    ((64,),      0.001, 64,  "mlp-h64-lr001-b64"),
    ((64,),      0.01,  64,  "mlp-h64-lr01-b64"),
    ((128,),     0.001, 64,  "mlp-h128-lr001-b64"),
    ((128,),     0.01,  64,  "mlp-h128-lr01-b64"),
    ((128, 64),  0.001, 128, "mlp-h128x64-lr001-b128"),
    ((128, 64),  0.01,  128, "mlp-h128x64-lr01-b128"),
    ((128, 128),  0.01,  128, "mlp-h128x128-lr01-b128"),
]

for hidden, lr, bs, name in configs:
    train_and_log(hidden, lr, bs, name)

mlp-h64-lr001-b64: train_loss=0.0277 val_acc=0.9719
🏃 View run mlp-h64-lr001-b64 at: http://localhost:5000/#/experiments/1/runs/b9e6183d6c85483ea00848e60db27163
🧪 View experiment at: http://localhost:5000/#/experiments/1


/home/sriharsha/projects/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (30) reached and the optimization hasn't converged yet.
  warnings.warn(


mlp-h64-lr01-b64: train_loss=0.0609 val_acc=0.9655
🏃 View run mlp-h64-lr01-b64 at: http://localhost:5000/#/experiments/1/runs/205685136f2c419c8dc48ff40a1075ef
🧪 View experiment at: http://localhost:5000/#/experiments/1
mlp-h128-lr001-b64: train_loss=0.0207 val_acc=0.9786
🏃 View run mlp-h128-lr001-b64 at: http://localhost:5000/#/experiments/1/runs/47549104ce504e6aa418c85b3b37a360
🧪 View experiment at: http://localhost:5000/#/experiments/1
mlp-h128-lr01-b64: train_loss=0.0553 val_acc=0.9684
🏃 View run mlp-h128-lr01-b64 at: http://localhost:5000/#/experiments/1/runs/d80dddbd19e647658f0e957c3a3ca256
🧪 View experiment at: http://localhost:5000/#/experiments/1


/home/sriharsha/projects/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (30) reached and the optimization hasn't converged yet.
  warnings.warn(


mlp-h128x64-lr001-b128: train_loss=0.0139 val_acc=0.9775
🏃 View run mlp-h128x64-lr001-b128 at: http://localhost:5000/#/experiments/1/runs/ac148b46cccb474ba51960ab06eeb11e
🧪 View experiment at: http://localhost:5000/#/experiments/1


/home/sriharsha/projects/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (30) reached and the optimization hasn't converged yet.
  warnings.warn(


mlp-h128x64-lr01-b128: train_loss=0.0346 val_acc=0.9744
🏃 View run mlp-h128x64-lr01-b128 at: http://localhost:5000/#/experiments/1/runs/3d5c59ecd5ab48a88fe3d605ca6c0893
🧪 View experiment at: http://localhost:5000/#/experiments/1


/home/sriharsha/projects/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (30) reached and the optimization hasn't converged yet.
  warnings.warn(


mlp-h128x128-lr01-b128: train_loss=0.0507 val_acc=0.9728
🏃 View run mlp-h128x128-lr01-b128 at: http://localhost:5000/#/experiments/1/runs/251f99cf2ada4fe6b89fb5b245185ce7
🧪 View experiment at: http://localhost:5000/#/experiments/1


In [6]:
runs_df = mlflow.search_runs(experiment_names=["mnist-mlp"], order_by=["metrics.val_accuracy DESC"])
print(runs_df[["run_id","tags.mlflow.runName","params.hidden_layer_sizes",
                "params.learning_rate_init","metrics.train_loss","metrics.val_accuracy"]].head(6))

                             run_id     tags.mlflow.runName  \
0  47549104ce504e6aa418c85b3b37a360      mlp-h128-lr001-b64   
1  ac148b46cccb474ba51960ab06eeb11e  mlp-h128x64-lr001-b128   
2  3d5c59ecd5ab48a88fe3d605ca6c0893   mlp-h128x64-lr01-b128   
3  251f99cf2ada4fe6b89fb5b245185ce7  mlp-h128x128-lr01-b128   
4  b9e6183d6c85483ea00848e60db27163       mlp-h64-lr001-b64   
5  d80dddbd19e647658f0e957c3a3ca256       mlp-h128-lr01-b64   

  params.hidden_layer_sizes params.learning_rate_init  metrics.train_loss  \
0                    (128,)                     0.001            0.020748   
1                 (128, 64)                     0.001            0.013905   
2                 (128, 64)                      0.01            0.034617   
3                (128, 128)                      0.01            0.050672   
4                     (64,)                     0.001            0.027742   
5                    (128,)                      0.01            0.055331   

   metrics.val_acc

## For Batch size experiment 

In [7]:
new_configs = [ 
    ((128, 64),  0.01, 64, "mlp-h128x64-lr01-b64"),
    ((128,), 0.001, 128, "mlp-h128-lr001-b128"),
]

In [8]:

for hidden, lr, bs, name in new_configs:
    train_and_log(hidden, lr, bs, name)

/home/sriharsha/projects/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (30) reached and the optimization hasn't converged yet.
  warnings.warn(


mlp-h128x64-lr01-b64: train_loss=0.0469 val_acc=0.9735
🏃 View run mlp-h128x64-lr01-b64 at: http://localhost:5000/#/experiments/1/runs/d78c036b1f0f4dc098f860d3f5393fbf
🧪 View experiment at: http://localhost:5000/#/experiments/1


/home/sriharsha/projects/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (30) reached and the optimization hasn't converged yet.
  warnings.warn(


mlp-h128-lr001-b128: train_loss=0.0106 val_acc=0.9791
🏃 View run mlp-h128-lr001-b128 at: http://localhost:5000/#/experiments/1/runs/f92fdb5468c94aff99d21c1573b73fa7
🧪 View experiment at: http://localhost:5000/#/experiments/1
